# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering

This notebook converts the W05 content archetypes into a ranked, human-review action queue. It uses observed signals only and does not claim that any action will cause a performance change.

**Source:** `work/outputs/content_archetypes_clustered.parquet` created by W05.

The playbook is intentionally decision-support: it prioritizes review work; a person decides what to change.

## 1. Ranked actions + reason codes

The queue ranks content for human review using the W05 archetype/action mapping plus transparent signal checks. Higher priority means the observed pattern is more decision-relevant, not that the recommended action is guaranteed to improve performance.

**Reason codes**

- `STP` — stale: older content and/or long time since update
- `VIS` — low visibility: low observed impressions
- `OPP` — search opportunity: meaningful search demand with weaker observed position/CTR
- `PRO` — protect: comparatively strong observed CTR/position
- `RARE` — rare archetype: small cluster, so treat as a narrow pattern
- `AMB` — ambiguous: low confidence / weak cluster assignment, requires manual review first
- `MISS` — material feature missingness or imputation dependency

**Action meanings**: Protect = avoid unnecessary changes; Improve = targeted optimization review; Rewrite = deeper content/freshness review; Monitor = watch without immediate change; Review = manual diagnosis before choosing an action.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("..")
CANDIDATES = [
    BASE / "outputs" / "content_archetypes_clustered.parquet",
    BASE / "outputs" / "content_level_model_dataset.parquet",
]

DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "W05 parquet output not found. Run W05 first and keep its output under work/outputs/."
    )

df = pd.read_parquet(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)

required = [
    "content_hash_id", "cluster",
    "search_volume", "impressions_90d", "ctr_90d",
    "avg_position_90d", "engagement_rate",
    "content_age_days", "days_since_update",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required W05 columns: {missing}")

print("Required W05 fields present: PASS")

### Action map used for this run

The current W05 interpretation maps Cluster 0 to **Low-Visibility Established Content → Improve**, Cluster 1 to **Stale Underperforming Content → Rewrite**, and Cluster 2 to **High-CTR Efficient Niche Content → Protect**. Cluster 2 is a rare pattern, so the queue applies an explicit rare-cluster review gate rather than treating it as a broad portfolio rule.

In [ ]:
action_map = {
    0: {"archetype": "Low-Visibility Established Content", "action": "Improve"},
    1: {"archetype": "Stale Underperforming Content", "action": "Rewrite"},
    2: {"archetype": "High-CTR Efficient Niche Content", "action": "Protect"},
}

# Validate that every observed cluster has an explicit mapping; never silently invent one.
observed_clusters = sorted(pd.Series(df["cluster"]).dropna().astype(int).unique().tolist())
missing_actions = [c for c in observed_clusters if c not in action_map]
if missing_actions:
    raise ValueError(f"No action mapping supplied for observed clusters: {missing_actions}")

print("Observed clusters:", observed_clusters)
print("Action map coverage: PASS")

## 2. Intended use and limits

A content/SEO team can use this queue to decide what to inspect first across a large content inventory. The queue is **not** a publishing system and does not automatically rewrite, delete, redirect, or protect pages.

The analysis is limited to the available structured snapshot: content characteristics, freshness, search visibility, and engagement. It does not inspect article semantics and does not establish causality. A recommendation such as Rewrite means the observed pattern is a candidate for human review, not that rewriting will improve traffic or rankings.

The W05 model is one row per content item, uses eight core features, and excludes IDs, query breadth, and future/trend labels from the clustering feature matrix.

In [ ]:
core_features = [
    "search_volume", "word_count", "content_age_days", "days_since_update",
    "impressions_90d", "ctr_90d", "avg_position_90d", "engagement_rate"
]

# Do not treat missing position as a real ranking value in the action logic.
for col in ["ctr_90d", "avg_position_90d", "engagement_rate"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["avg_position_90d"] = df["avg_position_90d"].replace(0, np.nan)

cluster_counts = df["cluster"].value_counts().sort_index()
cluster_pct = (cluster_counts / len(df) * 100).round(2)
print("Cluster sizes:")
display(pd.DataFrame({"n": cluster_counts, "pct": cluster_pct}))

print("Note: action rules use observed fields, while cluster membership comes from W05.")

## 3. Human review + the no-go list

Before acting on any row, a person should confirm the page's business importance, search intent, current content quality, recent changes, and whether the observed pattern is still true. Cluster membership is not a substitute for editorial judgment.

**No-go list**

- Do not auto-publish, auto-rewrite, or auto-delete content from this queue.
- Do not treat a rare cluster as a universal strategy.
- Do not infer a causal effect from cluster membership.
- Do not act on a row whose key signals are materially missing without checking the underlying data.
- Do not expose client names, domains, URLs, or raw/private queries in exported artifacts.
- Do not treat `avg_position_90d = 0` as a genuine ranking.

In [ ]:
# Signal thresholds are portfolio-relative and derived from this dataset, not universal SEO rules.
Q = df[["search_volume", "impressions_90d", "ctr_90d", "avg_position_90d", "engagement_rate", "content_age_days", "days_since_update"]].quantile([0.25, 0.50, 0.75])

P25 = Q.loc[0.25]
P50 = Q.loc[0.50]
P75 = Q.loc[0.75]

cluster_size_pct = (cluster_counts / len(df) * 100).to_dict()
rare_threshold_pct = 1.0

print("Portfolio medians:")
display(P50.to_frame("median"))
print(f"Rare-cluster threshold: < {rare_threshold_pct:.1f}% of content")

### Queue construction

Priority is a transparent score, not a probability. It combines action urgency with observed signal strength and then applies safety gates. The score is only for ordering the review queue.

In [ ]:
work = df.copy()

work["archetype"] = work["cluster"].map(lambda x: action_map.get(int(x), {}).get("archetype", "Review"))
work["base_action"] = work["cluster"].map(lambda x: action_map.get(int(x), {}).get("action", "Review"))
work["cluster_pct"] = work["cluster"].map(cluster_size_pct)

work["reason_codes"] = ""
work["priority_score"] = 0.0

# Staleness signal
stale_flag = (
    (work["days_since_update"] >= P75["days_since_update"]) |
    (work["content_age_days"] >= P75["content_age_days"])
).fillna(False)

# Visibility / opportunity signals
low_visibility_flag = (work["impressions_90d"] <= P25["impressions_90d"]).fillna(False)
search_opportunity_flag = (
    (work["search_volume"] >= P75["search_volume"]) &
    (work["avg_position_90d"] >= P50["avg_position_90d"])
).fillna(False)

# Protect signal
protect_flag = (
    (work["ctr_90d"] >= P75["ctr_90d"]) &
    (work["avg_position_90d"] <= P25["avg_position_90d"])
).fillna(False)

# Missingness / diagnostic gate
key_signal_missing = work[["search_volume", "impressions_90d", "ctr_90d", "avg_position_90d", "engagement_rate"]].isna().sum(axis=1) > 0

ambiguous_flag = pd.Series(False, index=work.index)
if "dist_to_own_centroid" in work.columns:
    ambiguous_cut = work["dist_to_own_centroid"].quantile(0.95)
    ambiguous_flag = work["dist_to_own_centroid"] >= ambiguous_cut

rare_flag = work["cluster_pct"] < rare_threshold_pct

# Transparent points for queue ordering
work.loc[work["base_action"] == "Rewrite", "priority_score"] += 4
work.loc[work["base_action"] == "Improve", "priority_score"] += 3
work.loc[work["base_action"] == "Protect", "priority_score"] += 1
work.loc[stale_flag, "priority_score"] += 3
work.loc[low_visibility_flag, "priority_score"] += 2
work.loc[search_opportunity_flag, "priority_score"] += 2
work.loc[protect_flag, "priority_score"] += 2
work.loc[rare_flag, "priority_score"] += 1
work.loc[ambiguous_flag, "priority_score"] += 2

# Reason codes
reasons = []
for idx, row in work.iterrows():
    codes = []
    if stale_flag.loc[idx]: codes.append("STP")
    if low_visibility_flag.loc[idx]: codes.append("VIS")
    if search_opportunity_flag.loc[idx]: codes.append("OPP")
    if protect_flag.loc[idx]: codes.append("PRO")
    if rare_flag.loc[idx]: codes.append("RARE")
    if ambiguous_flag.loc[idx]: codes.append("AMB")
    if key_signal_missing.loc[idx]: codes.append("MISS")
    reasons.append("+".join(codes) if codes else "PROFILE")
work["reason_codes"] = reasons

# Safety gates: ambiguous or material-missing rows become Review before any action.
work["final_action"] = work["base_action"]
work.loc[ambiguous_flag | key_signal_missing, "final_action"] = "Review"
work.loc[rare_flag & (work["final_action"] == "Protect"), "final_action"] = "Review"

# Human-check priority tier
work["priority_tier"] = pd.cut(
    work["priority_score"],
    bins=[-np.inf, 3, 6, np.inf],
    labels=["Monitor", "Review soon", "Review first"]
)

queue_cols = [
    "content_hash_id", "cluster", "archetype", "base_action", "final_action",
    "priority_tier", "priority_score", "reason_codes", "cluster_pct",
    "search_volume", "impressions_90d", "ctr_90d", "avg_position_90d",
    "engagement_rate", "content_age_days", "days_since_update"
]

queue = work[queue_cols].sort_values(
    ["priority_score", "impressions_90d"],
    ascending=[False, True]
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

queue = queue[
    ["rank"] + [c for c in queue.columns if c != "rank"]
]

display(queue.head(20))

## 4. Monitoring / retrain triggers

The playbook should be revisited when the portfolio, data coverage, or model behavior changes materially. These are monitoring triggers, not proof that retraining will improve recommendations.

In [ ]:
monitoring = pd.DataFrame([
    {
        "trigger": "Feature drift",
        "check": "Quarterly or after a material data-pipeline change, compare medians/quantiles for core features against the W05 reference snapshot.",
        "response": "Review preprocessing and consider rerunning the clustering pipeline if distributions materially shift."
    },
    {
        "trigger": "Cluster-size drift",
        "check": "Recompute cluster share. Flag a large change in a cluster's portfolio share (for example, >50% relative change) as a review trigger.",
        "response": "Inspect whether the archetype meaning still holds before using the action map."
    },
    {
        "trigger": "Separation deterioration",
        "check": "Recalculate silhouette on a comparable sample. A material drop from the reference should trigger an audit.",
        "response": "Inspect preprocessing, feature behavior, and K selection before publishing a new queue."
    },
    {
        "trigger": "Data coverage change",
        "check": "Track missingness in position, CTR, and engagement fields; also verify that no-data position remains encoded as missing.",
        "response": "Recheck imputation and leakage assumptions."
    },
    {
        "trigger": "New content or lifecycle change",
        "check": "Large influx of new pages, migrations, or major publishing changes can shift age/freshness distributions.",
        "response": "Reprofile clusters before reusing the existing action map."
    },
])

display(monitoring)

### Current monitoring snapshot

This cell records the current portfolio statistics that a future rerun can compare with.

In [ ]:
current_snapshot = pd.DataFrame({
    "metric": [
        "rows", "unique content", "clusters",
        "median search_volume", "median impressions_90d",
        "median ctr_90d", "median avg_position_90d",
        "median content_age_days", "median days_since_update"
    ],
    "value": [
        len(work),
        work["content_hash_id"].nunique(),
        work["cluster"].nunique(),
        work["search_volume"].median(),
        work["impressions_90d"].median(),
        work["ctr_90d"].median(),
        work["avg_position_90d"].median(),
        work["content_age_days"].median(),
        work["days_since_update"].median(),
    ]
})
display(current_snapshot)

## 5. Exports for the paper

The main export is a ranked review queue plus a compact summary of actions and reasons. Exports contain hashed content IDs and metrics only; no client names, domains, URLs, or raw queries.

In [ ]:
OUTPUT_DIR = BASE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

queue_path = OUTPUT_DIR / "content_action_queue.csv"
summary_path = OUTPUT_DIR / "content_action_summary.csv"
markdown_path = OUTPUT_DIR / "content_action_playbook.md"

summary = (
    queue.groupby(["archetype", "final_action", "priority_tier"], dropna=False)
    .agg(
        content_items=("content_hash_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr_90d", "median"),
        median_position=("avg_position_90d", "median"),
        median_days_since_update=("days_since_update", "median"),
    )
    .reset_index()
    .sort_values(["priority_tier", "content_items"], ascending=[True, False])
)

queue.to_csv(queue_path, index=False)
summary.to_csv(summary_path, index=False)

# Keep the paper-facing markdown compact and descriptive.
lines = [
    "# Content Action Playbook",
    "",
    f"Source rows: {len(work):,}. Unique content items: {work['content_hash_id'].nunique():,}.",
    "",
    "This queue is decision-support based on observed content, freshness, search, and engagement signals. It does not establish causal effects.",
    "",
    "## Ranked action summary",
    "",
]

for _, r in summary.iterrows():
    lines.append(
        f"- {r['archetype']} → {r['final_action']} — {int(r['content_items']):,} items; "
        f"median impressions {r['median_impressions']:.1f}, median CTR {r['median_ctr']:.3f}, "
        f"median position {r['median_position']:.2f}; priority {r['priority_tier']}."
    )

lines += [
    "",
    "## Review gates",
    "",
    "Ambiguous assignments, material missingness, and rare protect clusters are routed to human Review before action.",
    "",
    "## Limits",
    "",
    "The queue is a prioritization aid, not an automatic content-change system. It cannot prove that updating, rewriting, or protecting a page will improve future performance.",
]

markdown_path.write_text("\n".join(lines), encoding="utf-8")

print("Saved:")
print(queue_path.resolve())
print(summary_path.resolve())
print(markdown_path.resolve())
display(summary)

## Self-check

The checks below focus on reproducibility and claim safety.

In [ ]:
checks = []
checks.append(("W05 output loaded", len(df) > 0))
checks.append(("One row per content id", df["content_hash_id"].nunique() == len(df)))
checks.append(("All observed clusters have actions", not missing_actions))
checks.append(("No client-name/domain columns exposed", not any(t in c.lower() for c in df.columns for t in ["client_name", "domain", "brand", "url"])) )
checks.append(("No raw query column exposed", not any("query" in c.lower() for c in queue.columns)) )
checks.append(("No zero-position treated as valid rank", not (df["avg_position_90d"] == 0).any()))
checks.append(("Queue is ranked", queue["rank"].is_monotonic_increasing and len(queue) == len(df)))
checks.append(("Rare protect clusters routed to Review", ((rare_flag & (work["base_action"] == "Protect")) <= (work["final_action"] == "Review")).all()))

check_df = pd.DataFrame(checks, columns=["check", "passed"])
display(check_df)

if not check_df["passed"].all():
    raise AssertionError("One or more W07 self-checks failed. Review the table above.")

print("All automated W07 checks PASS.")
print("Manual check: run the notebook from a fresh kernel and confirm the exported queue matches the current W05 snapshot.")

### Interpretation note

The most important output is the **ranked review queue**, not an automatic action decision. The cluster names and mappings come from W05's observed profiles; this notebook adds a transparent prioritization layer and safety gates.